In [0]:
%sql
CREATE OR REPLACE TABLE banking.gold.branch_performance AS


WITH customer_branch AS (
    SELECT
        c.customer_id,
        c.branch_code
    FROM banking.silver.customers c
)

, account_agg AS (
    SELECT 
        a.customer_id,
        COUNT(a.account_id) AS total_accounts,
        SUM(a.balance) AS total_balance
    FROM banking.silver.accounts a
    GROUP BY a.customer_id
)

, txn_agg AS (
    SELECT 
        a.customer_id,
        COUNT(t.txn_id) AS total_transactions,
        SUM(t.amount) AS total_transaction_amount
    FROM banking.silver.transactions t
    LEFT JOIN banking.silver.accounts a
        ON a.account_id = t.account_id
    GROUP BY a.customer_id
)

SELECT 
    b.branch_code,
    b.branch_name,
    COUNT(DISTINCT cb.customer_id) AS total_customers,
    SUM(aa.total_accounts) AS total_accounts,
    SUM(aa.total_balance) AS total_deposits,
    SUM(ta.total_transactions) AS total_transactions,
    SUM(ta.total_transaction_amount) AS total_transaction_amount

FROM banking.silver.branches b

LEFT JOIN customer_branch cb
    ON b.branch_code = cb.branch_code

LEFT JOIN account_agg aa
    ON cb.customer_id = aa.customer_id

LEFT JOIN txn_agg ta
    ON cb.customer_id = ta.customer_id

GROUP BY 
b.branch_code, 
b.branch_name

In [0]:
count = spark.sql("""
    SELECT 
        COUNT(*) AS cnt
    FROM banking.gold.branch_performance
    """).collect()[0]["cnt"]

dbutils.notebook.exit(str(count))